# Check an ECB exchange-rate series across the Easter holiday

When a rate stays the same, dropping duplicates by value can erase a genuine observation. This example reads the GBP-per-euro series from a fixed Seiche snapshot and checks which dates the ECB actually published between 28 March and 2 April 2024. It uses the Python standard library. Run the two code cells in order.

The European Central Bank supplies the reference rates. The values express pounds per euro. The snapshot contains daily reference observations for information purposes. [ECB series](https://data.ecb.europa.eu/data/datasets/EXR/EXR.D.GBP.EUR.SP00.A) · [Seiche snapshot and attribution](https://huggingface.co/datasets/liquilens/seiche-ecb-reference-fx/tree/6a69269e2a5ce104defe87681203705ccc4876db) · [ECB reuse terms](https://www.ecb.europa.eu/services/using-our-site/disclaimer/html/index.en.html)

In [ ]:
import csv
import hashlib
import io
from decimal import Decimal
from urllib.request import urlopen

revision = "6a69269e2a5ce104defe87681203705ccc4876db"
url = f"https://huggingface.co/datasets/liquilens/seiche-ecb-reference-fx/resolve/{revision}/data/ecb_reference_fx_daily.csv"
with urlopen(url, timeout=30) as response:
    snapshot = response.read()
if hashlib.sha256(snapshot).hexdigest() != "d2c2ab02a8352c72cc539d420052ae8f09b98abbeacc4415cfbdacef5f2389ad":
    raise ValueError("The downloaded file differs from the reviewed snapshot.")


In [ ]:
rows = csv.DictReader(io.StringIO(snapshot.decode("utf-8-sig")))
observed = sorted(
    (row["date"], Decimal(row["value_per_eur"]))
    for row in rows
    if row["currency"] == "GBP" and "2024-03-28" <= row["date"] <= "2024-04-02"
)
expected = [("2024-03-28", Decimal("0.8551")), ("2024-04-02", Decimal("0.8551"))]
assert observed == expected, f"Unexpected observation dates or values: {observed}"
for day, value in observed:
    print(f"{day}   {value} GBP per EUR")


Both dates carry the same value. The four dates between them have no published row in this series. For an import or a replay, identify each observation by its series and date. Forward-filling would introduce four values the ECB did not publish.

Check the same interval directly at the ECB or explore the source conventions in Seiche. [ECB observations for this interval](https://data-api.ecb.europa.eu/service/data/EXR/D.GBP.EUR.SP00.A?startPeriod=2024-03-28&endPeriod=2024-04-02&format=csvdata) · [Seiche FX reference ledger](https://seiche.info/markets/forex/)